# Extração de CDI da B3 e IBOV do Yahoo Finance

Este notebook extrai:

- CDI a partir do FTP público da B3/Cetip
- IBOV a partir do Yahoo Finance (`^BVSP`)

Notas importantes:

- A página oficial da B3 informa que o arquivo diário segue o padrão `AAAAMMDD.txt`.
- A mesma página informa que o valor vem sem vírgula, por exemplo `000002320`, que deve ser interpretado como `23,20%`.
- O diretório `MediaCDI` abaixo é uma inferência prática amplamente usada para o FTP público da Cetip/B3. Se a B3 alterar a estrutura do FTP, basta ajustar a constante `FTP_DIR`.

In [8]:
# Se necessario, descomente:
# %pip install yfinance

In [9]:
from datetime import datetime
from ftplib import FTP
from io import BytesIO
from pathlib import Path

import pandas as pd
import yfinance as yf

In [10]:
DATA_INICIO = "2026-03-12"
DATA_FIM = datetime.today().strftime("%Y-%m-%d")

IBOV_TICKER = "^BVSP"

FTP_HOST = "ftp.cetip.com.br"
FTP_DIR = "MediaCDI"

SAIDA_DIR = Path("../data")
SAIDA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Funções auxiliares para o CDI na B3

In [11]:
def nome_arquivo_cdi(data_referencia):
    data = pd.Timestamp(data_referencia)
    return f"{data.strftime('%Y%m%d')}.txt"


def parse_taxa_di_bruta(texto):
    valor = texto.strip()
    if not valor:
        raise ValueError("Arquivo do CDI vazio.")

    if not valor.isdigit():
        raise ValueError(f"Conteudo inesperado para taxa CDI: {valor!r}")

    return int(valor) / 100


def baixar_taxa_cdi_b3(data_referencia, ftp_host=FTP_HOST, ftp_dir=FTP_DIR, timeout=30):
    arquivo = nome_arquivo_cdi(data_referencia)
    buffer = BytesIO()

    with FTP(host=ftp_host, timeout=timeout) as ftp:
        ftp.login()
        ftp.cwd(ftp_dir)
        ftp.retrbinary(f"RETR {arquivo}", buffer.write)

    conteudo = buffer.getvalue().decode("utf-8", errors="ignore").strip()
    taxa_percentual = parse_taxa_di_bruta(conteudo)

    return {
        "data": pd.Timestamp(data_referencia).normalize(),
        "cdi_percentual_aa": taxa_percentual,
        "arquivo_fonte": arquivo,
        "fonte": "B3 FTP",
    }


def baixar_serie_cdi_b3(data_inicio=DATA_INICIO, data_fim=DATA_FIM):
    datas = pd.date_range(data_inicio, data_fim, freq="D")
    registros = []
    falhas = []

    for data_ref in datas:
        try:
            registros.append(baixar_taxa_cdi_b3(data_ref))
        except Exception as exc:
            falhas.append({"data": pd.Timestamp(data_ref).normalize(), "erro": str(exc)})

    serie = pd.DataFrame(registros).sort_values("data").reset_index(drop=True)
    falhas_df = pd.DataFrame(falhas).sort_values("data").reset_index(drop=True)
    return serie, falhas_df

## 2. Teste de um dia específico do CDI

In [12]:
baixar_taxa_cdi_b3("2026-03-20")

{'data': Timestamp('2026-03-20 00:00:00'),
 'cdi_percentual_aa': 14.65,
 'arquivo_fonte': '20260320.txt',
 'fonte': 'B3 FTP'}

## 3. Baixar série do CDI

In [13]:
cdi_df, cdi_falhas = baixar_serie_cdi_b3(DATA_INICIO, DATA_FIM)
cdi_df.head()

,data,cdi_percentual_aa,arquivo_fonte,fonte
0,2026-03-12,14.9,20260312.txt,B3 FTP
1,2026-03-13,14.9,20260313.txt,B3 FTP
2,2026-03-16,14.9,20260316.txt,B3 FTP
3,2026-03-17,14.9,20260317.txt,B3 FTP
4,2026-03-18,14.9,20260318.txt,B3 FTP


In [14]:
cdi_falhas.head(20)

,data,erro
0,2026-03-14,550 Failed to open file.
1,2026-03-15,550 Failed to open file.
2,2026-03-21,550 Failed to open file.
3,2026-03-22,550 Failed to open file.
4,2026-03-24,550 Failed to open file.


## 4. Baixar IBOV do Yahoo Finance

In [15]:
def baixar_ibov_yahoo(ticker=IBOV_TICKER, data_inicio=DATA_INICIO, data_fim=DATA_FIM):
    df = yf.download(
        ticker,
        start=data_inicio,
        end=data_fim,
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=False,
    )

    if df.empty:
        raise ValueError(f"Nenhum dado retornado para o ticker {ticker}")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index().rename(columns={"Date": "data"})
    df["ticker"] = ticker
    return df

In [16]:
ibov_df = baixar_ibov_yahoo()
ibov_df.tail()

Price,data,Adj Close,Close,High,Low,Open,Volume,ticker
3,2026-03-17,180410.0,180410.0,182800.0,179850.0,179882.0,9332600,^BVSP
4,2026-03-18,179640.0,179640.0,181551.0,179576.0,180409.0,9750200,^BVSP
5,2026-03-19,180271.0,180271.0,181251.0,176296.0,179624.0,12560400,^BVSP
6,2026-03-20,176219.0,176219.0,180305.0,175039.0,180262.0,15348900,^BVSP
7,2026-03-23,181932.0,181932.0,182973.0,176221.0,176221.0,9874700,^BVSP


## 5. Consolidar CDI e IBOV

In [17]:
ibov_base = ibov_df[["data", "Open", "High", "Low", "Close", "Adj Close", "Volume", "ticker"]].copy()
ibov_base = ibov_base.rename(
    columns={
        "Open": "ibov_open",
        "High": "ibov_high",
        "Low": "ibov_low",
        "Close": "ibov_close",
        "Adj Close": "ibov_adj_close",
        "Volume": "ibov_volume",
        "ticker": "ibov_ticker",
    }
)

base_consolidada = ibov_base.merge(cdi_df, on="data", how="outer").sort_values("data").reset_index(drop=True)
base_consolidada.head()

,data,ibov_open,ibov_high,ibov_low,ibov_close,ibov_adj_close,ibov_volume,ibov_ticker,cdi_percentual_aa,arquivo_fonte,fonte
0,2026-03-12,183969.0,183992.0,178495.0,179285.0,179285.0,12075300,^BVSP,14.9,20260312.txt,B3 FTP
1,2026-03-13,179285.0,180996.0,177322.0,177653.0,177653.0,9525600,^BVSP,14.9,20260313.txt,B3 FTP
2,2026-03-16,177656.0,181255.0,177656.0,179875.0,179875.0,7370100,^BVSP,14.9,20260316.txt,B3 FTP
3,2026-03-17,179882.0,182800.0,179850.0,180410.0,180410.0,9332600,^BVSP,14.9,20260317.txt,B3 FTP
4,2026-03-18,180409.0,181551.0,179576.0,179640.0,179640.0,9750200,^BVSP,14.9,20260318.txt,B3 FTP


## 6. Métricas simples

In [18]:
base_consolidada["ibov_retorno_diario"] = base_consolidada["ibov_close"].pct_change()
base_consolidada["cdi_fator_diario_aprox"] = (1 + base_consolidada["cdi_percentual_aa"] / 100) ** (1 / 252) - 1
base_consolidada[["data", "ibov_close", "ibov_retorno_diario", "cdi_percentual_aa", "cdi_fator_diario_aprox"]].tail()

,data,ibov_close,ibov_retorno_diario,cdi_percentual_aa,cdi_fator_diario_aprox
3,2026-03-17,180410.0,0.002974,14.90,0.000551
4,2026-03-18,179640.0,-0.004268,14.90,0.000551
5,2026-03-19,180271.0,0.003513,14.65,0.000543
6,2026-03-20,176219.0,-0.022477,14.65,0.000543
7,2026-03-23,181932.0,0.032420,14.65,0.000543


## 7. Exportar resultados

In [19]:
arquivo_base = SAIDA_DIR / "cdi_b3_ibov_yahoo.csv"
arquivo_falhas = SAIDA_DIR / "cdi_b3_falhas.csv"

base_consolidada.to_csv(arquivo_base, index=False)
cdi_falhas.to_csv(arquivo_falhas, index=False)

arquivo_base, arquivo_falhas

(PosixPath('../data/cdi_b3_ibov_yahoo.csv'),
 PosixPath('../data/cdi_b3_falhas.csv'))